# Insurance Fraud



In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# Local VS Code: place CSVs alongside this notebook (or in the working directory).
# Kaggle: files are searched under /kaggle/input if not present locally.
def find_csv(filename):
    local = Path(filename)
    if local.exists():
        return local
    kaggle = list(Path('/kaggle/input').rglob(filename)) if Path('/kaggle/input').exists() else []
    if len(kaggle) == 1:
        return kaggle[0]
    raise FileNotFoundError(f'Cannot uniquely find {filename}; check notebook working directory / Kaggle inputs.')

train = pd.read_csv(find_csv('train.csv'))
test = pd.read_csv(find_csv('test.csv'))
assert {'claim_id', 'is_fraud', 'claim_amount'}.issubset(train.columns)
assert {'claim_id', 'claim_amount'}.issubset(test.columns)
assert train['claim_id'].is_unique and test['claim_id'].is_unique
print('Train:', train.shape, 'Test:', test.shape)
print('Fraud rate:', round(train['is_fraud'].mean(), 4))
display(train.head(3))

Train: (20000, 11) Test: (10000, 10)
Fraud rate: 0.1717


,claim_id,claim_amount,driver_age,previous_claim_count,vehicle_value,damage_severity,incident_type,repair_shop_id,phone_hash,address_id,is_fraud
0,C000001,254.29,42.0,2,11900.0,low,collision,Sb7a282e399,Pfd09f60f9e,A48656b5517,0
1,C000002,592.05,29.0,0,14600.0,low,collision,S9399315447,P8a7ff077bb,A53e94c25e5,0
2,C000007,980.14,58.0,0,17550.0,low,glass,S1bb582e486,P02246ac9cc,A84ed465d68,0


## 1. Feature preparation
Frequency maps are learned **only from the training portion** of each fold, and reused for validation/test. This avoids fitting frequency maps on validation rows. No target-derived statistics are used. The phone–address pair feature is aligned by position, without merging and changing indices.

In [2]:
TARGET = 'is_fraud'
DROP = ['claim_id', TARGET]
X = train.drop(columns=DROP).copy()
y = train[TARGET].astype(int).copy()
X_test_raw = test.drop(columns=['claim_id']).copy()
assert list(X.columns) == list(X_test_raw.columns)
cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
for df in (X, X_test_raw):
    for col in cat_cols:
        df[col] = df[col].fillna('Missing').astype(str)

ID_COLS = [c for c in ['phone_hash', 'repair_shop_id', 'address_id'] if c in X.columns]

def add_features(train_part, other_part, enhanced=False):
    a, b = train_part.copy(), other_part.copy()
    for df in (a, b):
        df['claim_vehicle_ratio'] = df['claim_amount'] / df['vehicle_value'].replace(0, np.nan)
        df['claim_vehicle_ratio'] = df['claim_vehicle_ratio'].replace([np.inf, -np.inf], np.nan)
    for col in ID_COLS:
        counts = a[col].value_counts(dropna=False)
        for df in (a, b):
            df[f'{col}_frequency'] = df[col].map(counts).fillna(0).astype(float)
    if enhanced:
        for df in (a, b):
            if 'phone_hash' in ID_COLS:
                df['high_phone_frequency'] = (df['phone_hash_frequency'] >= 3).astype(int)
            if 'address_id' in ID_COLS:
                df['high_address_frequency'] = (df['address_id_frequency'] >= 3).astype(int)
        if {'phone_hash', 'address_id'}.issubset(a.columns):
            pair_cols = ['phone_hash', 'address_id']
            pair_counts = a.groupby(pair_cols, dropna=False).size()
            for df in (a, b):
                keys = pd.MultiIndex.from_frame(df[pair_cols])
                df['phone_address_frequency'] = pair_counts.reindex(keys).fillna(0).to_numpy()

        if enhanced == "full":
            relationships = [
                ("phone_hash", "address_id", "phone_unique_addresses"),
                ("address_id", "phone_hash", "address_unique_phones"),
                ("phone_hash", "repair_shop_id", "phone_unique_repair_shops"),
                ("address_id", "repair_shop_id", "address_unique_repair_shops")
            ]

            for group_col, target_col, new_col in relationships:

                if group_col not in a.columns or target_col not in a.columns:
                    continue

                # Calculate unique counts from training data only
                counts = a.groupby(group_col)[target_col].nunique()

                # Apply to training and validation/test data
                for df in (a, b):
                    df[new_col] = (
                        df[group_col].map(counts).fillna(0)
                    )
    return a, b

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
amounts = train['claim_amount'].to_numpy(dtype=float)
fraud = y.to_numpy()

def score_predictions(probs, threshold=2500, budget=4000):
    expected = np.asarray(probs) * amounts
    eligible = np.flatnonzero(expected > threshold)
    selected = eligible[np.argsort(-expected[eligible])][:budget]
    score = float(np.sum(fraud[selected] * amounts[selected] - 2500))
    return {'ROC-AUC': roc_auc_score(fraud, probs), 'Investigated': len(selected),
            'Frauds Detected': int(fraud[selected].sum()), 'S Score': score,
            'Estimated Test S (rough)': score / 2}

def catboost_oof(params, enhanced):
    oof = np.zeros(len(X))
    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        a, b = add_features(X.iloc[tr], X.iloc[va], enhanced=enhanced)
        m = CatBoostClassifier(**params, loss_function='Logloss', verbose=False,
                               random_seed=42, cat_features=cat_cols, allow_writing_files=False)
        m.fit(a, y.iloc[tr])
        oof[va] = m.predict_proba(b)[:, 1]
    return oof

## 2. Verify the seven-feature pipeline

Force `enhanced=True` to include the 4 basic engineered features plus 3 phone/address features. The 4 additional relationship features are excluded (`enhanced="full"` is not used).

In [3]:
# Force exactly seven engineered features for all CV and final training.
use_enhanced = True
example_train, example_val = add_features(X.iloc[:100], X.iloc[100:110], enhanced=True)
new_features = [c for c in example_train.columns if c not in X.columns]
print('Engineered features (7 expected):', new_features)
assert len(new_features) == 7, f'Expected 7 engineered features, got {len(new_features)}'


Engineered features (7 expected): ['claim_vehicle_ratio', 'phone_hash_frequency', 'repair_shop_id_frequency', 'address_id_frequency', 'high_phone_frequency', 'high_address_frequency', 'phone_address_frequency']


## 3. CatBoost 4-model ensemble (5-fold out-of-fold validation)

In [4]:
configs = [
    dict(iterations=303, depth=6, learning_rate=0.05),
    dict(iterations=500, depth=5, learning_rate=0.05),
    dict(iterations=500, depth=7, learning_rate=0.05),
    dict(iterations=700, depth=6, learning_rate=0.03),
]
cat_oofs = [catboost_oof(p, enhanced=use_enhanced) for p in configs]
cat_ensemble_oof = np.mean(cat_oofs, axis=0)
cat_results = pd.DataFrame(
    {f'CatBoost {i+1}': score_predictions(p) for i, p in enumerate(cat_oofs)}
    | {'CatBoost 4-model ensemble': score_predictions(cat_ensemble_oof)}
).T.sort_values('S Score', ascending=False)
display(cat_results)

,ROC-AUC,Investigated,Frauds Detected,S Score,Estimated Test S (rough)
CatBoost 4-model ensemble,0.813376,2190.0,973.0,12242158.24,6121079.120
CatBoost 4,0.813369,2173.0,974.0,12213062.94,6106531.470
CatBoost 1,0.812381,2211.0,974.0,12212194.25,6106097.125
CatBoost 2,0.812699,2187.0,965.0,12161301.91,6080650.955
CatBoost 3,0.810608,2159.0,971.0,12150115.82,6075057.910


## 4. LightGBM and ensemble validation

Evaluate 70% CatBoost + 30% LightGBM using the same folds. Small CV differences may not generalise.

In [5]:
try:
    from lightgbm import LGBMClassifier
    has_lightgbm = True
except ImportError:
    has_lightgbm = False
    print('LightGBM unavailable. To test: %pip install lightgbm')

if has_lightgbm:
    oof_lgbm = np.zeros(len(X))
    for tr, va in skf.split(X, y):
        a, b = add_features(X.iloc[tr], X.iloc[va], enhanced=use_enhanced)
        for col in cat_cols:
            categories = pd.Index(a[col].unique())
            a[col] = pd.Categorical(a[col], categories=categories)
            b[col] = pd.Categorical(b[col], categories=categories)
        m = LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=15,
                           max_depth=6, min_child_samples=30, random_state=42, verbosity=-1)
        m.fit(a, y.iloc[tr], categorical_feature=cat_cols)
        oof_lgbm[va] = m.predict_proba(b)[:, 1]
    blends = {'CatBoost ensemble': cat_ensemble_oof, 'LightGBM': oof_lgbm}
    for weight in [0.1, 0.2, 0.3]:
        blends[f'CatBoost + {weight:.0%} LightGBM'] = ((1-weight)*cat_ensemble_oof + weight*oof_lgbm)
    blend_results = pd.DataFrame({k: score_predictions(v) for k,v in blends.items()}).T.sort_values('S Score', ascending=False)
    display(blend_results)
else:
    blend_results = None

,ROC-AUC,Investigated,Frauds Detected,S Score,Estimated Test S (rough)
CatBoost + 30% LightGBM,0.812643,2179.0,974.0,12337614.24,6168807.120
CatBoost + 20% LightGBM,0.813251,2181.0,975.0,12316742.37,6158371.185
CatBoost + 10% LightGBM,0.813489,2179.0,970.0,12254814.12,6127407.060
CatBoost ensemble,0.813376,2190.0,973.0,12242158.24,6121079.120
LightGBM,0.796219,2117.0,933.0,11845971.43,5922985.715


## 5. Train on all labelled claims and generate **two submissions**

The two CSV files contain exactly the required columns and no more than 2,000 investigations. Both use the same seven-feature engineering pipeline.

In [8]:
# Train the four CatBoost models on ALL labelled claims.
X_full, X_test = add_features(X, X_test_raw, enhanced=True)
assert len([c for c in X_full.columns if c not in X.columns]) == 7
cat_test_preds = []
for i, params in enumerate(configs, 1):
    model_final = CatBoostClassifier(
        **params, loss_function='Logloss', verbose=False,
        random_seed=42, cat_features=cat_cols, allow_writing_files=False
    )
    model_final.fit(X_full, y)
    cat_test_preds.append(model_final.predict_proba(X_test)[:, 1])
    print(f'Final CatBoost model {i}/4 fitted')
cat_probs = np.mean(cat_test_preds, axis=0)

# Train LightGBM on the SAME seven engineered features.
from lightgbm import LGBMClassifier
X_lgb_train, X_lgb_test = X_full.copy(), X_test.copy()
for col in cat_cols:
    categories = pd.Index(X_lgb_train[col].unique())
    X_lgb_train[col] = pd.Categorical(X_lgb_train[col], categories=categories)
    X_lgb_test[col] = pd.Categorical(X_lgb_test[col], categories=categories)

lgb_final = LGBMClassifier(
    n_estimators=500, learning_rate=0.03, num_leaves=15,
    max_depth=6, min_child_samples=30, random_state=42, verbosity=-1
)
lgb_final.fit(X_lgb_train, y, categorical_feature=cat_cols)
lgb_probs = lgb_final.predict_proba(X_lgb_test)[:, 1]

# Two distinct model predictions.
# Submission 1: 70% CatBoost + 30% LightGBM
probs_1 = 0.7 * cat_probs + 0.3 * lgb_probs

# Submission 2: 80% CatBoost + 20% LightGBM
probs_2 = 0.8 * cat_probs + 0.2 * lgb_probs

def save_submission(probs, filename):
    expected = np.asarray(probs) * test['claim_amount'].to_numpy(dtype=float)
    eligible = np.flatnonzero(expected > 2500)
    ranked = eligible[np.argsort(-expected[eligible])]
    selected = ranked[:2000]
    decisions = np.zeros(len(test), dtype=int)
    decisions[selected] = 1
    output = pd.DataFrame({'claim_id': test['claim_id'], 'investigate': decisions})
    assert output.shape == (len(test), 2)
    assert list(output.columns) == ['claim_id', 'investigate']
    assert output['claim_id'].is_unique and output['claim_id'].equals(test['claim_id'])
    assert output['investigate'].isin([0, 1]).all()
    assert output['investigate'].sum() <= 2000
    assert output.notna().all().all()
    output.to_csv(filename, index=False)
    print(f'{filename}: {int(output["investigate"].sum())} investigations')
    return output

submission_70_30 = save_submission(probs_1, 'submission_ensemble_70_30.csv')
submission_catboost = save_submission(probs_2, 'submission_ensemble_80_20.csv')
print('Different investigation decisions:', int((submission_70_30['investigate'] != submission_catboost['investigate']).sum()))
display(submission_70_30.head())


Final CatBoost model 1/4 fitted
Final CatBoost model 2/4 fitted
Final CatBoost model 3/4 fitted
Final CatBoost model 4/4 fitted
submission_ensemble_70_30.csv: 988 investigations
submission_ensemble_80_20.csv: 984 investigations
Different investigation decisions: 28


,claim_id,investigate
0,C000003,0
1,C000004,0
2,C000005,0
3,C000006,0
4,C000010,0


In [9]:
from xgboost import XGBClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

# Prepare data
X = train.drop(columns=["is_fraud", "claim_id"]).copy()
y = train["is_fraud"]

cat_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype(str)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_xgb = np.zeros(len(train))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):

    # Use seven engineered features
    X_tr, X_va = add_features(
        X.iloc[tr_idx],
        X.iloc[va_idx],
        enhanced=True
    )

    # Encode categorical variables
    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols])
    X_va[cat_cols] = encoder.transform(X_va[cat_cols])

    # Train XGBoost
    model_xgb = XGBClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        n_jobs=-1
    )

    model_xgb.fit(X_tr, y.iloc[tr_idx])

    oof_xgb[va_idx] = model_xgb.predict_proba(X_va)[:, 1]

    print(f"Fold {fold} completed")

# Calculate S score
amounts = train["claim_amount"].to_numpy()
fraud = y.to_numpy()

expected_value = oof_xgb * amounts

selected = np.where(expected_value > 2500)[0]
selected = selected[
    np.argsort(-expected_value[selected])
][:4000]

S_xgb = np.sum(
    fraud[selected] * amounts[selected] - 2500
)

print("\n===== XGBOOST RESULTS =====")
print(f"ROC-AUC: {roc_auc_score(y, oof_xgb):.4f}")
print(f"Investigated: {len(selected)}")
print(f"Frauds detected: {fraud[selected].sum()}")
print(f"S Score: ${S_xgb:,.2f}")
print(f"Estimated test S: ${S_xgb / 2:,.2f}")

Fold 1 completed
Fold 2 completed
Fold 3 completed
Fold 4 completed
Fold 5 completed

===== XGBOOST RESULTS =====
ROC-AUC: 0.7938
Investigated: 2020
Frauds detected: 932
S Score: $12,093,906.21
Estimated test S: $6,046,953.11
